In [0]:
%run "../../utils/00_config"

In [0]:
%run "../../utils/02_data_quality"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

stg_orders = spark.table(
    get_table("staging", "stg_wfm_orders")
)

print(f"Source rows: {stg_orders.count()}")

In [0]:
# Select latest player records
window_latest = (
    Window
    .partitionBy("player_id")
    .orderBy(
        F.col("observed_at").desc_nulls_last(),
        F.col("execution_id").desc(),
        F.col("order_id").desc()
    )
)

dim_player = (
    stg_orders
    .withColumn("_row_number", F.row_number().over(window_latest))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
'''
Dedup: Um jogador pode aparecer em várias ofertas e em diferentes execuções.
'''

dim_player = dim_player.select(
    "player_id",
    "player_name",
    "player_slug",
    "player_reputation",
    "platform",
    "crossplay",
    "locale",
    "player_status",
    "player_activity_type",
    "player_activity_details",
    "player_last_seen",
    F.col("execution_id").alias("source_execution_id"),
    F.col("observed_at").alias("player_observed_at")
)

In [0]:
# Data quality checks
check_required_fields(
    dim_player,
    ["player_id", "player_observed_at"]
)

check_unique_keys(dim_player, ["player_id"])

print(f"Dimension rows: {dim_player.count()}")

display(
    dim_player.orderBy("player_name")
)

In [0]:
from delta.tables import DeltaTable

target_table = get_table("intermediate", "dim_player")

if spark.catalog.tableExists(target_table):
    delta_table = DeltaTable.forName(spark, target_table)

    (
        delta_table.alias("target")
        .merge(
            dim_player.alias("source"),
            "target.player_id = source.player_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.player_observed_at >= target.player_observed_at
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        dim_player.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(target_table)
    )

print(f"Table materialized: {target_table}")